### Chunking EU AI Act PDF

#### Retrival

In [71]:
import fitz  # PyMuPDF
import pandas as pd

PDF_FILE = "data/eu_ai_act.pdf"
CHUNK_SIZE = 1000  # characters
CHUNK_OVERLAP = 200
PARQUET_OUT = "chunks/eu_ai_act_chunks.parquet"

# === Extract all text from the PDF
def extract_text_from_pdf(file_path):
    doc = fitz.open(file_path)
    full_text = ""
    for page in doc:
        full_text += page.get_text()
    return full_text

#### Chunking

In [73]:
# === Split text into overlapping chunks
def split_into_chunks(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# === Run the extraction + save
if __name__ == "__main__":
    try:
        print("Extracting text from EU AI Act PDF...")
        full_text = extract_text_from_pdf(PDF_FILE)

        print("Splitting into chunks...")
        chunks = split_into_chunks(full_text, CHUNK_SIZE, CHUNK_OVERLAP)

        df_chunks = pd.DataFrame({
            "source": "EU_AI_Act",
            "chunk_id": range(len(chunks)),
            "text": chunks
        })

        df_chunks.to_parquet(PARQUET_OUT, index=False)
        print(f"Saved {len(df_chunks)} chunks to {PARQUET_OUT}")
    except Exception as e:
        print(f"[PDF Error] Could not process {PDF_FILE}: {e}")

Extracting text from EU AI Act PDF...
Splitting into chunks...
Saved 750 chunks to chunks/eu_ai_act_chunks.parquet


#### Vector Store

In [53]:
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document
import pandas as pd
import os

# Config
datasets = {
    "faiss_eu_ai_act": "chunks/eu_ai_act_chunks.parquet"
}

embedding_function = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
output_base = "store_faiss"

# Step 1: Build and save FAISS vector stores
for store_name, parquet_path in datasets.items():
    print(f"Building FAISS store for: {store_name}")
    df = pd.read_parquet(parquet_path)

    if 'text' not in df.columns:
        print(f"Skipped {store_name} — no 'text' column found.")
        continue

    documents = [
        Document(
            page_content=row["text"],
            metadata={
                "chunk_id": row.get("chunk_id", i),
                "source": store_name
            }
        )
        for i, row in df.iterrows()
    ]

    vectorstore = FAISS.from_documents(documents, embedding=embedding_function)
    save_path = f"{output_base}/{store_name}_faiss"
    vectorstore.save_local(save_path)
    print(f"Saved FAISS store to: {save_path}\n")

# Step 2: Load all FAISS stores
loaded_stores = {}
for store_name in datasets:
    query_path = f"{output_base}/{store_name}_faiss"
    vectorstore = FAISS.load_local(
        query_path,
        embeddings=embedding_function,
        allow_dangerous_deserialization=True
    )
    loaded_stores[store_name] = vectorstore

# Step 3: Perform combined query across all stores
sample_query = "What is happening in AI and what are the legal obligations?"
top_k_per_store = 3

all_results = []
for store_name, store in loaded_stores.items():
    results = store.similarity_search_with_score(sample_query, k=top_k_per_store)
    for doc, score in results:
        doc.metadata["store"] = store_name
        all_results.append((doc, score))

# Step 4: Sort combined results by similarity score (lower is better)
all_results.sort(key=lambda x: x[1])  # score is distance

# Step 5: Display top N combined results
top_k_combined = 5
print(f"\nCombined top {top_k_combined} results for query:\n{sample_query}\n")

for i, (doc, score) in enumerate(all_results[:top_k_combined], 1):
    print(f"Result {i} | Source: {doc.metadata.get('store')} | Score: {round(score, 4)}")
    print(doc.page_content.strip()[:500])
    print("-" * 80)

Building FAISS store for: faiss_eu_ai_act
Saved FAISS store to: store_faiss/faiss_eu_ai_act_faiss


Combined top 5 results for query:
What is happening in AI and what are the legal obligations?

Result 1 | Source: faiss_eu_ai_act | Score: 0.49480000138282776
se of 
providing it, upon request, to the AI Office and the national competent authorities;
(b) draw up, keep up-to-date and make available information and documentation to providers of AI systems who intend to 
integrate the general-purpose AI model into their AI systems. Without prejudice to the need to observe and protect 
intellectual property rights and confidential business information or trade secrets in accordance with Union and 
national law, the information and documentation shall:
(i)
--------------------------------------------------------------------------------
Result 2 | Source: faiss_eu_ai_act | Score: 0.5669000148773193
eassessment at the earliest six months after that decision.
6.
The Commission shall ensure that 

#### Consume via API

In [ ]:
api_key = "" # Use env variable

In [68]:
import os
import requests
from flask import Flask, request, jsonify
from dotenv import load_dotenv

from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

# Load env
load_dotenv()

GEMINI_URL = (
    "https://generativelanguage.googleapis.com/v1beta/"
    f"models/gemini-2.5-flash:generateContent?key={api_key}"
)

# Config
FAISS_BASE_PATH = "store_faiss"
VECTOR_STORES = {
    "eu_ai_act": "faiss_eu_ai_act_faiss"
}

TOP_K_DEFAULT = 3

# Embeddings
embedding = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# Load FAISS stores once
stores = {}

for name, path in VECTOR_STORES.items():
    stores[name] = FAISS.load_local(
        os.path.join(FAISS_BASE_PATH, path),
        embeddings=embedding,
        allow_dangerous_deserialization=True
    )

# Flask app
app = Flask(__name__)

@app.route("/rag", methods=["POST"])
def rag():
    data = request.get_json()
    query = data.get("question")
    top_k = int(data.get("top_k", TOP_K_DEFAULT))

    if not query:
        return jsonify({"error": "Missing 'question'"}), 400

    retrieved = []

    for store_name, store in stores.items():
        results = store.similarity_search_with_score(query, k=top_k)
        for doc, score in results:
            retrieved.append({
                "source": store_name,
                "score": score,
                "text": doc.page_content
            })

    # Lower score = more similar
    retrieved.sort(key=lambda x: x["score"])

    # Build context
    context = "\n\n".join(
        f"[{r['source']}] {r['text']}"
        for r in retrieved
    )


    # Gemini prompt (minimal + strict)
    prompt = f"""
You are an expert assistant.

Answer the question using ONLY the context below.
If the context is insufficient, say:
"I cannot answer this based on the provided sources."

Context:
{context}

Question:
{query}

Answer:
""".strip()

    payload = {
        "contents": [
            {"parts": [{"text": prompt}]}
        ]
    }

    response = requests.post(
        GEMINI_URL,
        headers={"Content-Type": "application/json"},
        json=payload
    )

    if response.status_code != 200:
        return jsonify({"error": response.text}), response.status_code

    answer = response.json()["candidates"][0]["content"]["parts"][0]["text"]

    return jsonify({
        "question": query,
        "answer": answer,
        "sources": list({r["source"] for r in retrieved})
    })


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8200, debug=False, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8200
 * Running on http://192.168.1.133:8200
Press CTRL+C to quit
127.0.0.1 - - [14/Dec/2025 16:26:06] "POST /rag HTTP/1.1" 200 -
